# Synthetic-lethality pilot v2 analysis (Session 26)

v2 differs from v1 in two ways:

1. Pair candidates are pre-filtered to the **v15-non-essential set** (165 genes), so synth-lethality is defined for every sampled pair.
2. ESM-2 cosine bands are **calibrated against the actual pool distribution** (mean cos 0.930, std 0.038). Paralog signal sits at cos >= 0.97; baseline negative control sits at cos < 0.88.

Five categories:

| code | meaning | n_target |
|---|---|---:|
| A2_paralog_tight  | ESM-2 cos in [0.99, 1.00) | 24 |
| A2_paralog_loose  | ESM-2 cos in [0.97, 0.99) | 60 |
| B2_shared_substrate | one reactant shared in SBML | 54 |
| C2_shared_product | one product shared in SBML | 31 |
| D2_random_baseline | cos < 0.88 + no shared metabolite | 80 |

Total: 249 pairs. By construction every pair has both singles non-essential at v15.

Decision rules locked at start of v2:

| outcome | action |
|---|---|
| A2_tight rate >= 4x D2_random rate AND A2_tight rate >= 10 % | **SCIENTIFIC FINDING** — paralog redundancy detectable |
| A2_tight rate >= 2x D2_random rate AND A2_tight >= 5 % | **PARTIAL FINDING** — weak signal, name caveats |
| A2_tight rate <= 1.5x D2_random rate | **DOCUMENTED NULL** — methodology cannot detect synth-lethality at 0.5 s window |

In [ ]:
import pandas as pd
from pathlib import Path
REPO_ROOT = Path('.').resolve()
while not (REPO_ROOT / 'PROJECT_STATUS.md').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
df = pd.read_csv(REPO_ROOT / 'outputs/synthlet/pilot_v2_predictions.csv')
print(f'rows: {len(df)}  unique pairs: {df[["locus_a","locus_b"]].drop_duplicates().shape[0]}')
df.head()

In [ ]:
# 1. Per-category breakdown
g = (
    df.groupby('category')
      .agg(
          n=('is_synthetic_lethal', 'size'),
          n_synthlet=('is_synthetic_lethal', 'sum'),
          n_pair_essential=('pair_essential', 'sum'),
          n_eligible=('single_a_essential', lambda s: ((s == 0) & (df.loc[s.index, 'single_b_essential'] == 0)).sum()),
      )
)
g['rate_synthlet'] = g['n_synthlet'] / g['n']
g['rate_pair_essential'] = g['n_pair_essential'] / g['n']
print('Per-category breakdown (v2 — eligibility-filtered):')
print(g.to_string())

In [ ]:
# 2. Top synth-lethal predictions across categories
synth = df[df['is_synthetic_lethal'] == 1].copy()
synth = synth.sort_values('pair_confidence', ascending=False)
print(f'Total synthetic-lethal calls (v2): {len(synth)}')
if len(synth) > 0:
    for _, r in synth.head(20).iterrows():
        print(f"  {r['category']:24s} {r['locus_a']:18s} x {r['locus_b']:18s}  conf={r['pair_confidence']:.3f}  mode={r['pair_mode']}")
        print(f"      bio:  {str(r['biological_rationale'])[:100]}")
        print(f"      mech: {str(r['mechanism_summary'])[:130]}")

In [ ]:
# 3. Decision logic
rates = g['rate_synthlet'].to_dict()
rate_a_tight = float(rates.get('A2_paralog_tight', 0.0))
rate_a_loose = float(rates.get('A2_paralog_loose', 0.0))
rate_b = float(rates.get('B2_shared_substrate', 0.0))
rate_c = float(rates.get('C2_shared_product', 0.0))
rate_d = float(rates.get('D2_random_baseline', 0.0))

ratio_tight_over_baseline = (
    rate_a_tight / rate_d if rate_d > 0 else (float('inf') if rate_a_tight > 0 else 0.0)
)

print(f'A2_paralog_tight   (cos >= 0.99) : {rate_a_tight:.1%}')
print(f'A2_paralog_loose   (cos 0.97-0.99): {rate_a_loose:.1%}')
print(f'B2_shared_substrate              : {rate_b:.1%}')
print(f'C2_shared_product                : {rate_c:.1%}')
print(f'D2_random_baseline               : {rate_d:.1%}')
print(f'ratio (A2_tight / D2_random)     : {ratio_tight_over_baseline:.2f}x')

if rate_a_tight >= 0.10 and ratio_tight_over_baseline >= 4.0:
    decision = 'SCIENTIFIC_FINDING'
    rationale = (
        f'Tight-paralog synth-lethality rate {rate_a_tight:.1%} is at least '
        f'4x the random-baseline rate {rate_d:.1%} (ratio {ratio_tight_over_baseline:.1f}x). '
        f'The simulator separates paralog redundancy from random non-essential pairs.'
    )
elif rate_a_tight >= 0.05 and ratio_tight_over_baseline >= 2.0:
    decision = 'PARTIAL_FINDING'
    rationale = (
        f'Tight-paralog rate {rate_a_tight:.1%} is {ratio_tight_over_baseline:.1f}x '
        f'the random baseline {rate_d:.1%}. Signal is present but weak; n=24 tight '
        f'paralogs gives wide confidence intervals.'
    )
elif rate_a_tight <= rate_d * 1.5:
    decision = 'DOCUMENTED_NULL'
    rationale = (
        f'Tight-paralog rate {rate_a_tight:.1%} does not separate from random '
        f'baseline {rate_d:.1%}. The methodology cannot detect synth-lethality at the '
        f'0.5 s simulation window with v15 detector framework.'
    )
else:
    decision = 'INCONCLUSIVE'
    rationale = (
        f'Tight-paralog rate {rate_a_tight:.1%} is between the partial-finding and '
        f'null-finding thresholds. Sample size at n=24 is too small to distinguish.'
    )

print(f'\nDecision: {decision}')
print(f'Rationale: {rationale}')